# WIP Reconciliation, March 2026 Close

**Prepared by:** Benedict Daxell Santoso

**Date:** 13 August 2026

**Environment:** Databricks Free Edition, Unity Catalog, serverless compute

All queries ran on serverless compute. The workload is 1,780 transaction rows across three
files, with the heaviest operations being small aggregations and two-table joins. That is
several orders of magnitude below the point where distributed execution earns its
coordination overhead, so serverless is the appropriate choice on both startup latency and
cost. A dedicated cluster would add provisioning time without reducing runtime.

## Context

A professional services client reported that unbilled WIP totals in the staging
layer do not tie to the source system for March. Three symptoms were raised:

1. WIP overstated as of 31 March by an unquantified amount
2. March billed figures wrong on several projects
3. Some clients appearing twice with different regions

This notebook reconciles the staging report against the transaction source system,
quantifies each driver of the variance, and separates provable defects from
questions that require client confirmation.

## Sources

| File | Rows | Grain |
|---|---|---|
| `source_transactions.csv` | 1,780 | One row per transaction |
| `staging_wip_balance.csv` | 41 | One row per project, with exceptions |
| `project_dim.csv` | 77 | One row per project version, SCD Type 2 |

## Structure

The notebook runs in eight sections. Sections 1 to 3 create the schema, load the source
files into a raw layer, and validate the load. Section 4 profiles the three tables to
establish grain and locate data quality issues before any assumption is applied. Section 5
builds the clean layer, where each assumption appears as a named transformation. Section 6
reconciles the staging figure to a corrected figure. Section 7 restores the project the
staging join dropped. Section 8 summarises.

Findings and recommendations are in the accompanying write-up.

## 1. Environment setup

A dedicated schema keeps this work isolated from the default schema and gives
every table a single namespace under `workspace.wip_recon`. All queries below use
full three-part naming so each one is unambiguous about which catalog and schema
it reads from.

The volume acts as a landing zone. Source files were uploaded through the
workspace UI, since Free Edition runs on serverless compute with outbound network
access restricted and files cannot be pulled programmatically. They remain in the
volume unmodified, which keeps an unaltered copy of the source available for
verification independent of anything this notebook does to it.

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.wip_recon
COMMENT 'WIP reconciliation exercise, March 2026 close';

CREATE VOLUME IF NOT EXISTS workspace.wip_recon.landing;

## 2. Raw layer

The three files load into managed Delta tables prefixed `raw_`. These tables are
a faithful copy of what arrived and stay that way for the rest of the notebook.
No deduplication, no filtering, no correction.

That constraint is deliberate. The reconciliation below identifies duplicate
transactions and duplicated project rows in the staging report. Keeping the raw
tables unmodified is what demonstrates those defects arrived in the source data
rather than being introduced by transformation logic here. Every correction is
applied in a separate clean layer, where it is visible as a named step and can be
traced back to the assumption that motivated it.

Types are inferred at load and validated in the next cell. A production pipeline
would land raw as string and cast explicitly downstream, which prevents a change
in source formatting from silently altering how historical loads were parsed.

In [0]:
BASE = "/Volumes/workspace/wip_recon/landing"

files = {
    "raw_source_transactions": "source_transactions.csv",
    "raw_staging_wip_balance": "staging_wip_balance.csv",
    "raw_project_dim":         "project_dim.csv",
}

for table, filename in files.items():
    (spark.read
        .option("header", True)
        .option("inferSchema", True)
        .csv(f"{BASE}/{filename}")
        .write.mode("overwrite")
        .saveAsTable(f"workspace.wip_recon.{table}"))
    print(f"loaded {table}")

loaded raw_source_transactions
loaded raw_staging_wip_balance
loaded raw_project_dim


## 3. Load validation

Row counts are checked against the source files before any analysis begins. A
short count indicates a parsing failure, most likely an unescaped delimiter
inside a text field, which would otherwise surface much later as an unexplained
variance.

Data types are checked at the same time. Date columns are the specific risk. A
date parsed as string still compares without error but compares
lexicographically, so a period cutoff filter would return the wrong rows and
report success.

Inference resolved all date columns to `date` and `created_at_utc` to `timestamp`, so period comparisons operate on real date types rather than string ordering. No casting is required in the clean layer.

In [0]:
expected = {
    "raw_source_transactions": 1780,
    "raw_staging_wip_balance": 41,
    "raw_project_dim":         77,
}

for table, n in expected.items():
    actual = spark.table(f"workspace.wip_recon.{table}").count()
    print(f"{table:28} {actual:>6} expected {n:>6}  {'OK' if actual == n else 'MISMATCH'}")

print()
for table in expected:
    print(f"--- {table} ---")
    spark.table(f"workspace.wip_recon.{table}").printSchema()

raw_source_transactions        1780 expected   1780  OK
raw_staging_wip_balance          41 expected     41  OK
raw_project_dim                  77 expected     77  OK

--- raw_source_transactions ---
root
 |-- transaction_id: string (nullable = true)
 |-- project_id: string (nullable = true)
 |-- transaction_date: date (nullable = true)
 |-- posting_date: date (nullable = true)
 |-- transaction_type: string (nullable = true)
 |-- hours: double (nullable = true)
 |-- wip_amount: double (nullable = true)
 |-- wip_status: integer (nullable = true)
 |-- write_up_down: double (nullable = true)
 |-- billed_amount: double (nullable = true)
 |-- invoice_id: string (nullable = true)
 |-- created_at_utc: timestamp (nullable = true)

--- raw_staging_wip_balance ---
root
 |-- project_id: string (nullable = true)
 |-- project_name: string (nullable = true)
 |-- client_name: string (nullable = true)
 |-- region: string (nullable = true)
 |-- project_manager: string (nullable = true)
 |-- project_st

In [0]:
%sql
DESCRIBE HISTORY workspace.wip_recon.raw_source_transactions;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
0,2026-08-12T14:07:53.000Z,73474080362817,benedict.d.santoso@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.format.version"":""2.12.0"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(1888749321008363),2217d937-6197-4afe-bf3f-61b21619ec5f,0812-135803-lju2ax0v-v2n,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 1780, numOutputBytes -> 33612)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13


Version 0 is the only entry, recorded as CREATE OR REPLACE TABLE AS SELECT with isV1SaveAsTableOverwrite set true, reflecting the mode("overwrite") used in the load. Delta appends a new version on every subsequent write, so any later change to this table is visible here and the prior state stays queryable through time travel.

## 4. Profiling

Before applying any assumption, the three tables are profiled to establish grain, test key
uniqueness, and locate data quality issues. Each query below serves a specific question,
and the answers determine what the clean layer has to handle.

The order matters. Grain tests come first, because a table that does not hold the grain it
claims will produce wrong answers to every question asked of it afterward. Both the
transaction table and the staging report fail that test, in different ways.

| Query | Question |
|---|---|
| Q1, Q1b | Do the transaction table and the staging report hold one row per key? |
| Q1c | Where the staging report duplicates, which fields differ? |
| Q2, Q2b | Are the duplicate transactions identical, and what are they worth? |
| Q3 | What does each `wip_status` value look like, and does anything distinguish them? |
| Q4, Q4b | Do any transactions post outside the period they belong to, and in which direction? |
| Q5, Q5b | Do the three tables agree on which projects exist? |
| Q6, Q6b | How does the dimension handle change over time, and does the staging join respect it? |

In [0]:
%sql
-- Q1, grain and uniqueness test on transactions

SELECT
  COUNT(*)                                    AS total_rows,
  COUNT(DISTINCT transaction_id)              AS distinct_transaction_ids,
  COUNT(*) - COUNT(DISTINCT transaction_id)   AS row_count_minus_distinct_ids
FROM workspace.wip_recon.raw_source_transactions;

total_rows,distinct_transaction_ids,row_count_minus_distinct_ids
1780,1772,8


In [0]:
%sql
-- Q1b, staging grain test

SELECT
  COUNT(*)                              AS total_rows,
  COUNT(DISTINCT project_id)            AS distinct_projects,
  COUNT(*) - COUNT(DISTINCT project_id) AS excess_rows
FROM workspace.wip_recon.raw_staging_wip_balance;

total_rows,distinct_projects,excess_rows
41,39,2


In [0]:
%sql
-- Q1c, duplicated staging projects and which attributes differ

SELECT
  project_id, project_name, client_name, region,
  project_manager, project_status, wip_amount, billed_amount
FROM workspace.wip_recon.raw_staging_wip_balance
WHERE project_id IN (
  SELECT project_id
  FROM workspace.wip_recon.raw_staging_wip_balance
  GROUP BY project_id HAVING COUNT(*) > 1
)
ORDER BY project_id, client_name;

project_id,project_name,client_name,region,project_manager,project_status,wip_amount,billed_amount
PRJ-1010,Copperline Tax Advisory FY26,Copperline Energy,Southeast,Harper Moreau,Active,33586.0,20467.0
PRJ-1010,Copperline Tax Advisory FY26,Copperline Energy Holdings,West,Harper Moreau,Active,33586.0,20467.0
PRJ-1026,Ironbridge Tax Advisory FY26,Ironbridge Construction,Southeast,Harper Moreau,Active,49743.0,0.0
PRJ-1026,Ironbridge Tax Advisory FY26,Ironbridge Construction Group,Midwest,Harper Moreau,Active,49743.0,0.0


In [0]:
%sql
-- Q2, duplicate transaction_id detection, full attribute comparison

WITH dup_ids AS (
  SELECT transaction_id
  FROM workspace.wip_recon.raw_source_transactions
  GROUP BY transaction_id
  HAVING COUNT(*) > 1
)
SELECT t.*
FROM workspace.wip_recon.raw_source_transactions t
INNER JOIN dup_ids d
  ON t.transaction_id = d.transaction_id
ORDER BY t.transaction_id, t.created_at_utc;

transaction_id,project_id,transaction_date,posting_date,transaction_type,hours,wip_amount,wip_status,write_up_down,billed_amount,invoice_id,created_at_utc
TXN-50027,PRJ-1001,2026-03-19,2026-03-20,LABOR,6.0,1050.0,0,0.0,0.0,null,2026-03-20T13:38:00.000Z
TXN-50027,PRJ-1001,2026-03-19,2026-03-20,LABOR,6.0,1050.0,0,0.0,0.0,null,2026-03-20T13:38:00.000Z
TXN-50163,PRJ-1004,2026-03-26,2026-03-28,LABOR,14.0,1330.0,0,0.0,0.0,null,2026-03-28T13:56:00.000Z
TXN-50163,PRJ-1004,2026-03-26,2026-03-28,LABOR,14.0,1330.0,0,0.0,0.0,null,2026-03-28T13:56:00.000Z
TXN-50303,PRJ-1007,2026-03-26,2026-03-27,LABOR,12.0,2520.0,0,0.0,0.0,null,2026-03-27T18:09:00.000Z
TXN-50303,PRJ-1007,2026-03-26,2026-03-27,LABOR,12.0,2520.0,0,0.0,0.0,null,2026-03-27T18:09:00.000Z
TXN-50310,PRJ-1007,2026-03-11,2026-03-13,EXPENSE,0.0,1678.0,0,0.0,0.0,null,2026-03-13T13:12:00.000Z
TXN-50310,PRJ-1007,2026-03-11,2026-03-13,EXPENSE,0.0,1678.0,0,0.0,0.0,null,2026-03-13T13:12:00.000Z
TXN-50925,PRJ-1021,2026-03-03,2026-03-05,LABOR,12.0,2100.0,0,0.0,0.0,null,2026-03-05T18:38:00.000Z
TXN-50925,PRJ-1021,2026-03-03,2026-03-05,LABOR,12.0,2100.0,0,0.0,0.0,null,2026-03-05T18:38:00.000Z


In [0]:
%sql
-- Q2b, dollar value of the duplication

WITH dup_ids AS (
  SELECT transaction_id
  FROM workspace.wip_recon.raw_source_transactions
  GROUP BY transaction_id
  HAVING COUNT(*) > 1
),
ranked AS (
  SELECT
    t.*,
    ROW_NUMBER() OVER (PARTITION BY t.transaction_id ORDER BY t.created_at_utc) AS copy_rank
  FROM workspace.wip_recon.raw_source_transactions t
  INNER JOIN dup_ids d
    ON t.transaction_id = d.transaction_id
)
SELECT
  COUNT(*)             AS extra_copy_count,
  SUM(wip_amount)       AS extra_copy_wip_amount,
  SUM(write_up_down)    AS extra_copy_write_up_down,
  SUM(billed_amount)    AS extra_copy_billed_amount
FROM ranked
WHERE copy_rank > 1;

extra_copy_count,extra_copy_wip_amount,extra_copy_write_up_down,extra_copy_billed_amount
8,12392.0,0.0,0.0


In [0]:
%sql
-- Q3, wip_status profile

SELECT
  wip_status,
  COUNT(*)                                                             AS row_count,
  SUM(wip_amount)                                                      AS wip_sum,
  SUM(billed_amount)                                                   AS billed_sum,
  SUM(write_up_down)                                                   AS adjustment_sum,
  SUM(CASE WHEN invoice_id IS NULL THEN 1 ELSE 0 END)                  AS null_invoice_count,
  ROUND(SUM(CASE WHEN invoice_id IS NULL THEN 1 ELSE 0 END) / COUNT(*) * 100, 1) AS null_invoice_pct
FROM workspace.wip_recon.raw_source_transactions
GROUP BY wip_status
ORDER BY wip_status;

wip_status,row_count,wip_sum,billed_sum,adjustment_sum,null_invoice_count,null_invoice_pct
0,879,1732175.0,0.0,0.0,879,100.0
1,24,47179.0,0.0,0.0,24,100.0
2,877,1773446.0,1648069.0,-125377.0,0,0.0


In [0]:
%sql
-- Q4, March transaction dates with posting dates after 3/31

SELECT
  DATEDIFF(posting_date, transaction_date) AS posting_lag_days,
  COUNT(*)                                  AS row_count
FROM workspace.wip_recon.raw_source_transactions
GROUP BY DATEDIFF(posting_date, transaction_date)
ORDER BY posting_lag_days;

posting_lag_days,row_count
0,580
1,604
2,587
3,5
4,2
5,1
6,1


In [0]:
%sql
-- Q4b, mirror case check, February transactions posted into March

SELECT
  MIN(DATEDIFF(posting_date, transaction_date))                 AS min_lag_days,
  MAX(DATEDIFF(posting_date, transaction_date))                 AS max_lag_days,
  ROUND(AVG(DATEDIFF(posting_date, transaction_date)), 2)       AS avg_lag_days,
  PERCENTILE(DATEDIFF(posting_date, transaction_date), 0.5)     AS median_lag_days
FROM workspace.wip_recon.raw_source_transactions;

min_lag_days,max_lag_days,avg_lag_days,median_lag_days
0,6,1.02,1.0


In [0]:
%sql
-- Q5, project count reconciliation across all three tables

SELECT 'transactions' AS source_table, COUNT(DISTINCT project_id) AS distinct_project_count
FROM workspace.wip_recon.raw_source_transactions
UNION ALL
SELECT 'staging', COUNT(DISTINCT project_id)
FROM workspace.wip_recon.raw_staging_wip_balance
UNION ALL
SELECT 'project_dim', COUNT(DISTINCT project_id)
FROM workspace.wip_recon.raw_project_dim;

source_table,distinct_project_count
transactions,40
staging,39
project_dim,69


In [0]:
%sql
-- Q5b, anti-join, projects in transactions but not in the dimension

SELECT DISTINCT t.project_id
FROM workspace.wip_recon.raw_source_transactions t
LEFT JOIN workspace.wip_recon.raw_staging_wip_balance s
  ON t.project_id = s.project_id
WHERE s.project_id IS NULL;

project_id
PRJ-1013


The reciprocal check returns nothing. Every project in the staging report exists in the
transaction table, so the gap runs in one direction only: the staging report is missing a
project the source system has, not carrying one the source system does not.

In [0]:
%sql
-- Q6, SCD Type 2 version analysis

WITH multi_version AS (
  SELECT project_id
  FROM workspace.wip_recon.raw_project_dim
  GROUP BY project_id
  HAVING COUNT(*) > 1
)
SELECT
  d.project_id,
  d.project_dim_key,
  d.client_name,
  d.region,
  d.office,
  d.service_line,
  d.project_manager,
  d.billing_partner,
  d.effective_from,
  d.effective_to,
  d.is_current
FROM workspace.wip_recon.raw_project_dim d
INNER JOIN multi_version m
  ON d.project_id = m.project_id
ORDER BY d.project_id, d.effective_from;

project_id,project_dim_key,client_name,region,office,service_line,project_manager,billing_partner,effective_from,effective_to,is_current
PRJ-1003,4003,Bluepeak Logistics,Midwest,Chicago,Advisory,Harper Larsen,Sage Weber,2025-03-01,2025-11-30,0
PRJ-1003,4004,Bluepeak Logistics,Midwest,Charlotte,Assurance,Riley Novak,Morgan Okafor,2025-12-01,9999-12-31,1
PRJ-1008,4009,Atlas Freight Systems,West,Newark,Tax,Jordan Moreau,Skyler Haddad,2025-03-01,2025-11-30,0
PRJ-1008,4010,Atlas Freight Systems,West,Charlotte,Assurance,Reese Silva,Dana Ortiz,2025-12-01,9999-12-31,1
PRJ-1010,4012,Copperline Energy,Southeast,New York,Tax,Harper Moreau,Sage Weber,2025-06-01,2026-03-14,0
PRJ-1010,4013,Copperline Energy Holdings,West,Philadelphia,Advisory,Harper Moreau,Dana Patel,2026-03-15,9999-12-31,1
PRJ-1017,4019,Brightside Pharmacy Group,Northeast,Philadelphia,Advisory,Dana Ortiz,Logan Novak,2025-03-01,2025-11-30,0
PRJ-1017,4020,Brightside Pharmacy Group,Northeast,Charlotte,Advisory,Avery Nguyen,Blake Haddad,2025-12-01,9999-12-31,1
PRJ-1022,4025,Cascade Foods Group,Southeast,New York,Managed Services,Blake Costa,Blake Okafor,2025-03-01,2025-11-30,0
PRJ-1022,4026,Cascade Foods Group,Southeast,Chicago,Assurance,Quinn Byrne,Emerson Novak,2025-12-01,9999-12-31,1


In [0]:
%sql
-- Q6b, why only two of eight multi-version projects fanned 

WITH multi_version AS (
  SELECT project_id
  FROM workspace.wip_recon.raw_project_dim
  GROUP BY project_id HAVING COUNT(*) > 1
),
staging_counts AS (
  SELECT project_id, COUNT(*) AS staging_row_count
  FROM workspace.wip_recon.raw_staging_wip_balance
  GROUP BY project_id
)
SELECT
  d.project_id,
  MIN(d.effective_from)                          AS earliest_version_from,
  MAX(d.effective_from)                          AS latest_version_from,
  COUNT(*)                                       AS dimension_versions,
  COALESCE(s.staging_row_count, 0)               AS staging_rows,
  CASE WHEN MAX(d.effective_from)
            BETWEEN '2026-03-01' AND '2026-03-31'
       THEN 'version changed during March'
       ELSE 'version changed outside March' END  AS change_timing
FROM workspace.wip_recon.raw_project_dim d
INNER JOIN multi_version m ON d.project_id = m.project_id
LEFT JOIN staging_counts s ON d.project_id = s.project_id
GROUP BY d.project_id, s.staging_row_count
ORDER BY d.project_id;

project_id,earliest_version_from,latest_version_from,dimension_versions,staging_rows,change_timing
PRJ-1003,2025-03-01,2025-12-01,2,1,version changed outside March
PRJ-1008,2025-03-01,2025-12-01,2,1,version changed outside March
PRJ-1010,2025-06-01,2026-03-15,2,2,version changed during March
PRJ-1017,2025-03-01,2025-12-01,2,1,version changed outside March
PRJ-1022,2025-03-01,2025-12-01,2,1,version changed outside March
PRJ-1026,2025-06-01,2026-03-15,2,2,version changed during March
PRJ-1030,2025-03-01,2025-12-01,2,1,version changed outside March
PRJ-1038,2025-03-01,2025-12-01,2,1,version changed outside March


All eight multi-version projects appear in staging, but only two duplicate. Both are the
only two whose dimension version changed on 15 March, inside the reporting period. The six
that changed in December 2025 resolve to a single row each.

That rules out a missing version filter, which would have fanned all eight. The join
matches any dimension version whose validity window intersects the reporting period rather
than the single version valid at 31 March. Both rows for PRJ-1010 qualify: one valid
through 14 March, one from 15 March onward.

The distinction matters for the fix. Filtering on `is_current` resolves the duplication
but stamps today's client and region onto a March report. A point-in-time predicate
against the as-of date returns exactly one version per project and reproduces the same
result whenever it is re-run, which is what section 5 applies.

## 5. Clean layer

Each assumption is applied here as a named transformation, so any line in this layer can
be traced back to the decision that motivated it.

The governing rule is that the clean layer classifies, it does not delete. Rows excluded
from unbilled WIP are flagged rather than dropped, and the filtering happens in the
reconciliation. That keeps every excluded amount queryable, which is what allows the
bridge to disclose each exclusion as its own line rather than having amounts disappear
without trace.

| Table | Transformation |
|---|---|
| `clean_transactions` | Deduplicated, plus `is_within_cutoff` and `wip_classification` |
| `clean_project_dim` | Point-in-time predicate at 31 March |
| `clean_staging_wip` | One row per project, dimension attributes dropped |

The duplicate rows are byte-identical across all twelve source columns, so deduplication
is lossless and no row selection decision arises. The cutoff is an upper bound, not a
month filter: open WIP at 31 March is a cumulative balance that includes January and
February work still unbilled.

In [0]:
%sql
-- Build clean_transactions

CREATE OR REPLACE TABLE workspace.wip_recon.clean_transactions AS
SELECT
  transaction_id,
  project_id,
  transaction_date,
  posting_date,
  transaction_type,
  hours,
  wip_amount,
  wip_status,
  write_up_down,
  billed_amount,
  invoice_id,
  created_at_utc,
  -- Upper bound only, not a month filter. Open WIP at 3/31 is a cumulative balance and
  -- includes January and February work that is still unbilled.
  posting_date <= DATE'2026-03-31' AS is_within_cutoff,
  -- No ELSE. A wip_status value outside 0/1/2 should surface as NULL here, not get
  -- silently absorbed into one of the three named buckets.
  CASE wip_status
    WHEN 0 THEN 'open'
    WHEN 1 THEN 'held'
    WHEN 2 THEN 'relieved'
  END AS wip_classification
FROM (
  -- The eight duplicate pairs are byte-identical across all twelve source columns,
  -- including created_at_utc. SELECT DISTINCT over the full column set is therefore a
  -- lossless dedup: there is no row selection decision to make, because there is no
  -- attribute on which the two copies of a pair differ.
  SELECT DISTINCT
    transaction_id,
    project_id,
    transaction_date,
    posting_date,
    transaction_type,
    hours,
    wip_amount,
    wip_status,
    write_up_down,
    billed_amount,
    invoice_id,
    created_at_utc
  FROM workspace.wip_recon.raw_source_transactions
);

num_affected_rows,num_inserted_rows


In [0]:
%sql
-- Build clean_project_dim

CREATE OR REPLACE TABLE workspace.wip_recon.clean_project_dim AS
SELECT
  project_dim_key,
  project_id,
  project_name,
  client_name,
  region,
  office,
  service_line,
  project_manager,
  billing_partner,
  project_status,
  budget_amount,
  start_date,
  effective_from,
  effective_to,
  is_current
FROM workspace.wip_recon.raw_project_dim
WHERE effective_from <= DATE'2026-03-31'
  AND effective_to   >= DATE'2026-03-31';


num_affected_rows,num_inserted_rows


In [0]:
%sql
-- Build clean_staging_wip

CREATE OR REPLACE TABLE workspace.wip_recon.clean_staging_wip AS
SELECT DISTINCT
  project_id,
  wip_amount,
  write_up_down,
  billed_amount
FROM workspace.wip_recon.raw_staging_wip_balance;

num_affected_rows,num_inserted_rows


### Validation

Six checks. The last carries the most weight, since it confirms `wip_classification`
partitioned the table rather than losing rows.

In [0]:
%sql
-- Validation 1, clean_transactions row count

SELECT
  (SELECT COUNT(DISTINCT transaction_id) FROM workspace.wip_recon.raw_source_transactions) AS raw_dedup_row_count,
  (SELECT COUNT(*) FROM workspace.wip_recon.clean_transactions)                             AS clean_transactions_row_count,
  (SELECT COUNT(DISTINCT transaction_id) FROM workspace.wip_recon.raw_source_transactions)
    = (SELECT COUNT(*) FROM workspace.wip_recon.clean_transactions)                          AS dedup_matches_clean,
  (SELECT COUNT(*) FROM workspace.wip_recon.clean_transactions) = 1772                       AS matches_target_1772;

raw_dedup_row_count,clean_transactions_row_count,dedup_matches_clean,matches_target_1772
1772,1772,true,true


In [0]:
%sql
-- Validation 2, clean_project_dim rows and distinct projects

SELECT
  COUNT(*)                              AS clean_project_dim_row_count,
  COUNT(DISTINCT project_id)            AS clean_project_dim_distinct_projects,
  COUNT(*) = COUNT(DISTINCT project_id) AS rows_equal_distinct_projects,
  COUNT(*) = 51                         AS matches_target_51
FROM workspace.wip_recon.clean_project_dim;

clean_project_dim_row_count,clean_project_dim_distinct_projects,rows_equal_distinct_projects,matches_target_51
51,51,true,true


In [0]:
%sql
-- Validation 2b, count of projects excluded by the point-in-time filter

SELECT
  COUNT(DISTINCT project_id) AS excluded_project_count,
  COUNT(*)                    AS excluded_row_count
FROM workspace.wip_recon.raw_project_dim
WHERE project_id NOT IN (SELECT project_id FROM workspace.wip_recon.clean_project_dim);

excluded_project_count,excluded_row_count
18,18


In [0]:
%sql
-- Validation 2c, excluded project detail, status and validity windows

SELECT
  d.project_id,
  d.project_status,
  d.effective_from,
  d.effective_to,
  d.is_current
FROM workspace.wip_recon.raw_project_dim d
WHERE d.project_id NOT IN (SELECT project_id FROM workspace.wip_recon.clean_project_dim)
ORDER BY d.project_id, d.effective_from;

project_id,project_status,effective_from,effective_to,is_current
PRJ-903,Closed,2024-03-01,2025-11-28,0
PRJ-904,Closed,2024-03-01,2025-10-28,0
PRJ-905,Closed,2024-12-01,2025-07-28,0
PRJ-906,Closed,2024-11-01,2025-07-28,0
PRJ-907,Closed,2024-08-01,2025-12-28,0
PRJ-908,Closed,2024-06-01,2025-11-28,0
PRJ-915,Closed,2024-05-01,2025-09-28,0
PRJ-916,Closed,2024-02-01,2025-09-28,0
PRJ-917,Closed,2024-04-01,2025-10-28,0
PRJ-918,Closed,2024-09-01,2025-09-28,0


In [0]:
%sql
-- Validation 2d, no active project falls in the excluded set
-- PRJ-1013 is absent from raw_project_dim entirely rather than excluded by the point-in-time filter, so it does not appear here either way. Expect zero.

SELECT
  COUNT(DISTINCT t.project_id) = 0 AS no_active_project_in_excluded_set,
  COUNT(DISTINCT t.project_id)      AS active_project_in_excluded_set_count
FROM (
  SELECT project_id FROM workspace.wip_recon.raw_source_transactions
  UNION
  SELECT project_id FROM workspace.wip_recon.raw_staging_wip_balance
) t
WHERE t.project_id IN (
  SELECT project_id
  FROM workspace.wip_recon.raw_project_dim
  WHERE project_id NOT IN (SELECT project_id FROM workspace.wip_recon.clean_project_dim)
);

no_active_project_in_excluded_set,active_project_in_excluded_set_count
true,0


In [0]:
%sql
-- Validation 3, clean_staging_wip row count

SELECT
  COUNT(*)      AS clean_staging_wip_row_count,
  COUNT(*) = 39  AS matches_target_39
FROM workspace.wip_recon.clean_staging_wip;

clean_staging_wip_row_count,matches_target_39
39,true


In [0]:
%sql
-- Validation 4, wip_classification partitions clean_transactions

SELECT
  wip_status,
  wip_classification,
  COUNT(*)            AS row_count,
  SUM(wip_amount)      AS wip_sum,
  SUM(billed_amount)   AS billed_sum,
  SUM(write_up_down)   AS adjustment_sum
FROM workspace.wip_recon.clean_transactions
GROUP BY wip_status, wip_classification
ORDER BY wip_status;

wip_status,wip_classification,row_count,wip_sum,billed_sum,adjustment_sum
0,open,871,1719783.0,0.0,0.0
1,held,24,47179.0,0.0,0.0
2,relieved,877,1773446.0,1648069.0,-125377.0


## 6. Reconciliation

Two independent pieces. The billing tests prove a defect outright and depend on no
assumption. The bridge applies the assumptions from section 5 and walks the staging figure
to a reconciled figure.

| Query | Proves |
|---|---|
| Q7, Q7b | Whether staging computes billed as gross plus adjustment or gross minus adjustment |
| Q8, Q8b | Whether the `write_up_down` column itself ties, which isolates where the error sits |
| Bridge | Every driver of the WIP variance, quantified and summing exactly |

In [0]:
%sql
-- Q7, billing formula test, per project

WITH status2_march AS (
  SELECT
    project_id,
    SUM(wip_amount)    AS gross_wip_relieved,
    SUM(write_up_down) AS march_adjustment_sum
  FROM workspace.wip_recon.clean_transactions
  WHERE wip_classification = 'relieved'
    AND posting_date BETWEEN DATE'2026-03-01' AND DATE'2026-03-31'
  GROUP BY project_id
)
SELECT
  s.project_id,
  s.gross_wip_relieved,
  s.march_adjustment_sum,
  s.gross_wip_relieved + s.march_adjustment_sum AS formula_gross_plus_adjustment,
  s.gross_wip_relieved - s.march_adjustment_sum AS formula_gross_minus_adjustment,
  w.billed_amount                                AS staging_billed_amount,
  (s.gross_wip_relieved + s.march_adjustment_sum) - w.billed_amount AS variance_gross_plus_adjustment,
  (s.gross_wip_relieved - s.march_adjustment_sum) - w.billed_amount AS variance_gross_minus_adjustment
FROM status2_march s
INNER JOIN workspace.wip_recon.clean_staging_wip w
  ON s.project_id = w.project_id
ORDER BY s.project_id;

project_id,gross_wip_relieved,march_adjustment_sum,formula_gross_plus_adjustment,formula_gross_minus_adjustment,staging_billed_amount,variance_gross_plus_adjustment,variance_gross_minus_adjustment
PRJ-1002,7898.0,-1026.0,6872.0,8924.0,8924.0,-2052.0,0.0
PRJ-1003,14037.0,-1505.0,12532.0,15542.0,15542.0,-3010.0,0.0
PRJ-1004,2580.0,-198.0,2382.0,2778.0,2778.0,-396.0,0.0
PRJ-1005,13422.0,-2856.0,10566.0,16278.0,16278.0,-5712.0,0.0
PRJ-1006,17326.0,-2074.0,15252.0,19400.0,19400.0,-4148.0,0.0
PRJ-1007,18537.0,-449.0,18088.0,18986.0,18986.0,-898.0,0.0
PRJ-1008,9291.0,-307.0,8984.0,9598.0,9598.0,-614.0,0.0
PRJ-1009,11711.0,-180.0,11531.0,11891.0,11891.0,-360.0,0.0
PRJ-1010,19393.0,-1074.0,18319.0,20467.0,20467.0,-2148.0,0.0
PRJ-1011,5828.0,-97.0,5731.0,5925.0,5925.0,-194.0,0.0


In [0]:
%sql
-- Q7b, billing formula test, rollup

WITH status2_march AS (
  SELECT
    project_id,
    SUM(wip_amount)    AS gross_wip_relieved,
    SUM(write_up_down) AS march_adjustment_sum
  FROM workspace.wip_recon.clean_transactions
  WHERE wip_classification = 'relieved'
    AND posting_date BETWEEN DATE'2026-03-01' AND DATE'2026-03-31'
  GROUP BY project_id
),
compared AS (
  SELECT
    s.project_id,
    (s.gross_wip_relieved + s.march_adjustment_sum) - w.billed_amount AS variance_plus,
    (s.gross_wip_relieved - s.march_adjustment_sum) - w.billed_amount AS variance_minus
  FROM status2_march s
  INNER JOIN workspace.wip_recon.clean_staging_wip w
    ON s.project_id = w.project_id
)
SELECT
  COUNT(*)                                                   AS project_count,
  SUM(CASE WHEN variance_plus = 0 THEN 1 ELSE 0 END)          AS ties_gross_plus_adjustment,
  SUM(CASE WHEN variance_minus = 0 THEN 1 ELSE 0 END)         AS ties_gross_minus_adjustment,
  MAX(ABS(variance_plus))                                     AS max_abs_variance_plus,
  MAX(ABS(variance_minus))                                    AS max_abs_variance_minus
FROM compared;

project_count,ties_gross_plus_adjustment,ties_gross_minus_adjustment,max_abs_variance_plus,max_abs_variance_minus
31,0,31,5712.0,0.0


In [0]:
%sql
-- Q8, write_up_down control test, per project

WITH source_march_adjustments AS (
  SELECT
    project_id,
    SUM(write_up_down) AS source_march_write_up_down
  FROM workspace.wip_recon.clean_transactions
  WHERE wip_classification = 'relieved'
  AND posting_date BETWEEN DATE'2026-03-01' AND DATE'2026-03-31'
  GROUP BY project_id
)
SELECT
  s.project_id,
  s.source_march_write_up_down,
  w.write_up_down                                    AS staging_write_up_down,
  s.source_march_write_up_down - w.write_up_down      AS variance
FROM source_march_adjustments s
INNER JOIN workspace.wip_recon.clean_staging_wip w
  ON s.project_id = w.project_id
ORDER BY s.project_id;

project_id,source_march_write_up_down,staging_write_up_down,variance
PRJ-1002,-1026.0,-1026.0,0.0
PRJ-1003,-1505.0,-1505.0,0.0
PRJ-1004,-198.0,-198.0,0.0
PRJ-1005,-2856.0,-2856.0,0.0
PRJ-1006,-2074.0,-2074.0,0.0
PRJ-1007,-449.0,-449.0,0.0
PRJ-1008,-307.0,-307.0,0.0
PRJ-1009,-180.0,-180.0,0.0
PRJ-1010,-1074.0,-1074.0,0.0
PRJ-1011,-97.0,-97.0,0.0


In [0]:
%sql
-- Q8b, write_up_down control test, rollup

WITH source_march_adjustments AS (
  SELECT
    project_id,
    SUM(write_up_down) AS source_march_write_up_down
  FROM workspace.wip_recon.clean_transactions
  WHERE wip_classification = 'relieved'
  AND posting_date BETWEEN DATE'2026-03-01' AND DATE'2026-03-31'
  GROUP BY project_id
),
compared AS (
  SELECT
    s.source_march_write_up_down - w.write_up_down AS variance
  FROM source_march_adjustments s
  INNER JOIN workspace.wip_recon.clean_staging_wip w
    ON s.project_id = w.project_id
)
SELECT
  COUNT(*)                                          AS project_count,
  SUM(CASE WHEN variance = 0 THEN 1 ELSE 0 END)      AS ties_exactly,
  MAX(ABS(variance))                                 AS max_abs_variance
FROM compared;

project_count,ties_exactly,max_abs_variance
31,31,0.0


### WIP bridge

Starts from `clean_staging_wip`. Four lines come off it. Three are filters already built
into `clean_transactions` as classification columns. The fourth is not a filter at all:
the clean layer deduplicated upstream, so the duplicate adjustment has to be computed as
the raw-to-clean difference or it disappears from the bridge without a trace.

Every exclusion line is scoped to projects present in `clean_staging_wip`. That scoping
is load-bearing. PRJ-1013 carries held WIP that was never in the starting total, and
without the scoping it would be deducted from a balance it never entered, producing a
bridge that is internally inconsistent while still appearing to reconcile. Cell 29c
tests whether that case actually arises rather than assuming it does not.

| Line | Derived from |
|---|---|
| Staging total | `clean_staging_wip` |
| Held rows excluded | `wip_classification = 'held'` |
| Late postings excluded | `is_within_cutoff = false` |
| Duplicates removed | Raw minus clean, open and within cutoff |
| Projects restored | Open, within cutoff, absent from staging |

In [0]:
%sql
-- Build recon_wip_bridge

CREATE OR REPLACE TABLE workspace.wip_recon.recon_wip_bridge AS
WITH staging_total AS (
  SELECT SUM(wip_amount) AS amount
  FROM workspace.wip_recon.clean_staging_wip
),
held_excluded AS (
  SELECT SUM(wip_amount) AS amount
  FROM workspace.wip_recon.clean_transactions
  WHERE wip_classification = 'held'
    AND project_id IN (SELECT project_id FROM workspace.wip_recon.clean_staging_wip)
),
late_excluded AS (
  SELECT SUM(wip_amount) AS amount
  FROM workspace.wip_recon.clean_transactions
  WHERE is_within_cutoff = false
    AND project_id IN (SELECT project_id FROM workspace.wip_recon.clean_staging_wip)
),
duplicates_removed AS (
  -- Not a filter. The clean layer already deduplicated, so this has to be computed as
  -- the raw-to-clean difference, restricted to the open, within-cutoff population that
  -- feeds the reconciled balance, and to the projects staging already covers.
  SELECT
    (SELECT SUM(wip_amount)
     FROM workspace.wip_recon.raw_source_transactions
     WHERE wip_status = 0
       AND posting_date <= DATE'2026-03-31'
       AND project_id IN (SELECT project_id FROM workspace.wip_recon.clean_staging_wip))
    -
    (SELECT SUM(wip_amount)
     FROM workspace.wip_recon.clean_transactions
     WHERE wip_classification = 'open'
       AND is_within_cutoff = true
       AND project_id IN (SELECT project_id FROM workspace.wip_recon.clean_staging_wip))
    AS amount
),
projects_restored AS (
  -- Generalized rather than hardcoded to PRJ-1013: any project in clean_transactions
  -- that clean_staging_wip does not carry gets its open, within-cutoff balance restored.
  SELECT SUM(wip_amount) AS amount
  FROM workspace.wip_recon.clean_transactions
  WHERE wip_classification = 'open'
    AND is_within_cutoff = true
    AND project_id NOT IN (SELECT project_id FROM workspace.wip_recon.clean_staging_wip)
)
SELECT 1 AS line_order, 'Staging total, starting balance' AS line_label, amount AS line_amount FROM staging_total
UNION ALL
SELECT 2, 'Held rows excluded', -amount FROM held_excluded
UNION ALL
SELECT 3, 'Late postings excluded', -amount FROM late_excluded
UNION ALL
SELECT 4, 'Duplicates removed', -amount FROM duplicates_removed
UNION ALL
SELECT 5, 'Projects absent from staging restored (PRJ-1013)', amount FROM projects_restored;

num_affected_rows,num_inserted_rows


In [0]:
%sql
-- Bridge with running total

SELECT
  line_order,
  line_label,
  line_amount,
  SUM(line_amount) OVER (
    ORDER BY line_order
    ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
  ) AS running_total
FROM workspace.wip_recon.recon_wip_bridge
ORDER BY line_order;

line_order,line_label,line_amount,running_total
1,"Staging total, starting balance",1726306.0,1726306.0
2,Held rows excluded,-42539.0,1683767.0
3,Late postings excluded,-30100.0,1653667.0
4,Duplicates removed,-12392.0,1641275.0
5,Projects absent from staging restored (PRJ-1013),48408.0,1689683.0


In [0]:
%sql
-- Bridge scoping diagnostic

WITH raw_counts AS (
  SELECT project_id, COUNT(*) AS raw_row_count
  FROM workspace.wip_recon.raw_source_transactions
  GROUP BY project_id
)
SELECT
  ct.project_id,
  SUM(CASE WHEN ct.wip_classification = 'held' THEN ct.wip_amount ELSE 0 END) AS held_wip_amount,
  SUM(CASE WHEN ct.is_within_cutoff = false THEN ct.wip_amount ELSE 0 END)    AS late_wip_amount,
  COUNT(*)                 AS clean_row_count,
  MAX(rc.raw_row_count)    AS raw_row_count
FROM workspace.wip_recon.clean_transactions ct
LEFT JOIN raw_counts rc
  ON rc.project_id = ct.project_id
WHERE ct.project_id NOT IN (SELECT project_id FROM workspace.wip_recon.clean_staging_wip)
GROUP BY ct.project_id;

project_id,held_wip_amount,late_wip_amount,clean_row_count,raw_row_count
PRJ-1013,4640.0,0.0,42,42


### Bridge assertions

The three exclusion lines assume no transaction contributes to more than one of them. A
held row that was also posted late would be deducted twice. A relieved row posted after
the cutoff would be deducted from a balance that never contained it, since staging's WIP
figure covers open and held rows only.

Neither case arises in this data, but the bridge depends on it silently. The cell below
converts the assumption into a check.

In [0]:
%sql
-- Bridge line mutual exclusivity

SELECT
  SUM(CASE WHEN wip_classification = 'held' AND is_within_cutoff = false THEN 1 ELSE 0 END)  AS held_and_late,
  SUM(CASE WHEN is_within_cutoff = false AND wip_classification <> 'open' THEN 1 ELSE 0 END) AS late_and_not_open
FROM workspace.wip_recon.clean_transactions;

held_and_late,late_and_not_open
0,0


## 7. Restoring projects absent from staging

Register entry 3 resolved to restoring PRJ-1013 at project level with dimension
attributes left unassigned. This section builds the reconciled project table and applies
that restoration.

The base table is `clean_staging_wip`, 39 projects. The merge adds any project that has
open, within-cutoff WIP in `clean_transactions` but no row in staging.

Written as a `MERGE` with a generalised source rather than an `INSERT` naming PRJ-1013.
The staging pipeline inner joins to `project_dim`, so any project without a master data
record is dropped silently and this will recur. A merge handles three missing projects
next month as readily as one this month. An insert hardcoded to a single ID solves today
and fails quietly the next time.

Dimension attributes are not populated. There is no defensible basis for deriving
`client_name` or `region` from transaction data, so the restored row carries a source
label instead and any client or region rollup will show it as unassigned.

This table applies the restoration only, not the three exclusions from the bridge. It
totals the staging figure plus the restored balance, not the reconciled figure.

In [0]:
%sql
-- Build recon_project_wip

CREATE OR REPLACE TABLE workspace.wip_recon.recon_project_wip AS
SELECT
  project_id,
  wip_amount AS reconciled_wip_amount,
  'staging'  AS source
FROM workspace.wip_recon.clean_staging_wip;

num_affected_rows,num_inserted_rows


In [0]:
%sql
-- MERGE, restore projects absent from staging

MERGE INTO workspace.wip_recon.recon_project_wip AS target
USING (
  SELECT
    project_id,
    SUM(wip_amount) AS reconciled_wip_amount
  FROM workspace.wip_recon.clean_transactions
  WHERE wip_classification = 'open'
    AND is_within_cutoff = true
    AND project_id NOT IN (SELECT project_id FROM workspace.wip_recon.clean_staging_wip)
  GROUP BY project_id
) AS source
ON target.project_id = source.project_id
WHEN NOT MATCHED THEN
  INSERT (project_id, reconciled_wip_amount, source)
  VALUES (source.project_id, source.reconciled_wip_amount, 'restored from transactions');

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
1,0,0,1


In [0]:
%sql
-- Merge verification

SELECT
  COUNT(*)                                                              AS project_count,
  SUM(CASE WHEN source = 'staging' THEN 1 ELSE 0 END)                   AS from_staging,
  SUM(CASE WHEN source = 'restored from transactions' THEN 1 ELSE 0 END) AS restored,
  SUM(reconciled_wip_amount)                                            AS total_wip_amount
FROM workspace.wip_recon.recon_project_wip;

project_count,from_staging,restored,total_wip_amount
40,39,1,1774714.0


### Table history after the merge

Two versions. Version 0 is the create, version 1 the merge.

Version 1 records the merge with `numSourceRows` and `numTargetRowsInserted` both at 1
and `numTargetRowsUpdated` at 0. The restoration inserted PRJ-1013 and modified no
existing balance, so no staging figure was silently overwritten. `readVersion` at 0
identifies the pre-restoration state, which stays queryable through time travel. The
report as staging produced it can be compared against the corrected version without
maintaining a second pipeline.

In [0]:
%sql
-- DESCRIBE HISTORY on recon_project_wip

DESCRIBE HISTORY workspace.wip_recon.recon_project_wip;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
3,2026-08-13T19:24:29.000Z,73474080362817,benedict.d.santoso@gmail.com,MERGE,"Map(predicate -> [""(project_id#17461 = project_id#17479)""], clusterBy -> [], matchedPredicates -> [], statsOnLoad -> true, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(1888749321008363),eb862db3-2e3f-4000-99e3-c80cb276b54b,0813-180605-5z4nrx1g-v2n,2,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 1, numTargetBytesAdded -> 1448, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 0, executionTimeMs -> 1636, materializeSourceTimeMs -> 17, numTargetRowsInserted -> 1, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 0, numTargetRowsUpdated -> 0, numOutputRows -> 1, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 1, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 1566)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
2,2026-08-13T19:23:43.000Z,73474080362817,benedict.d.santoso@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.format.version"":""2.12.0"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(1888749321008363),8dee5ce9-971c-4bc4-847d-93c44a85d69c,0813-180605-5z4nrx1g-v2n,1,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 2, numRemovedBytes -> 3095, numDeletionVectorsRemoved -> 0, numOutputRows -> 39, numOutputBytes -> 1647)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
1,2026-08-13T13:56:47.000Z,73474080362817,benedict.d.santoso@gmail.com,MERGE,"Map(predicate -> [""(project_id#11454 = project_id#11458)""], clusterBy -> [], matchedPredicates -> [], statsOnLoad -> true, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(2766257967363839),9334fd92-7ac7-425a-a2ad-05d94f722c1f,0813-134537-c581f88b-v2n,0,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 1, numTargetBytesAdded -> 1448, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 0, executionTimeMs -> 2550, materializeSourceTimeMs -> 46, numTargetRowsInserted -> 1, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 0, numTargetRowsUpdated -> 0, numOutputRows -> 1, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 1, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 2448)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
0,2026-08-13T13:46:07.000Z,73474080362817,benedict.d.santoso@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.format.version"":""2.12.0"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(2766257967363839),c74ce766-3c21-4bf5-88b7-b7c68606d0c8,0813-134537-c581f88b-v2n,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 39, numOutputBytes -> 1647)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13


## 8. Summary

Unbilled WIP at 31 March reconciles to 1,689,683 against a staging figure of 1,726,306.
The net variance of 36,623 nets four drivers, three overstating by 85,031 combined and one
understating by 48,408. March billing reconciles to 308,228 against a staging figure of
351,971.

Findings, proposed changes, and the assumptions requiring confirmation are set out in the
accompanying write-up.

## Structure

The notebook runs in eight sections. Sections 1 to 3 create the schema, load the source
files into a raw layer, and validate the load. Section 4 profiles the three tables to
establish grain and locate data quality issues before any assumption is applied. Section 5
builds the clean layer, where each assumption appears as a named transformation. Section 6
reconciles the staging figure to a corrected figure. Section 7 restores the project the
staging join dropped. Section 8 summarises.

Findings and recommendations are in the accompanying write-up.